# Задания: дерево решений и Random Forest: прогноз кликов

Работаем с `dataset/bycicles.csv`: одна строка — клетка рекламного отчёта, а не отдельный пользователь.
**Цель:** предсказать наличие хотя бы одного клика (`wCTR (%) > 0`, официальная метка соревнования) по данным, известным до показа.
Это актуальная бинарная задача курса, а не прогноз расхода из старой тетради.

План: официальное разделение → дерево → лес → отбор признаков → приёмы из Kaggle → итоговая таблица → SHAP.
Во всех обучаемых деревьях задаём `class_weight`. Сравниваем ROC AUC и BalancedAccuracy; рейтинг — среднее двух метрик.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier, BaggingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import ParameterGrid, train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, RocCurveDisplay

SEED = 42
N_JOBS = 4
ROOT = Path.cwd() if (Path.cwd() / "dataset").exists() else Path.cwd().parent
DATA_PATH = ROOT / "dataset/bycicles.csv"
OUT = ROOT / "seminars/trees_results"
OUT.mkdir(exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4)})
print("Python", platform.python_version(), "sklearn", sklearn.__version__)

## 1. Данные и утечки

`Клики`, `CTR (%)`, `wCTR (%)`, расход, показы и конверсии описывают результат рекламного показа.
Не используем их как вход. Явный список допустимых столбцов безопаснее удаления только целевого.
Удаляем дублирующий номер группы, пустую ссылку и постоянный тип площадки по смыслу полей.
Текст и заголовок пока считаем категориями; дату заменяем календарём. Масштабирование и PCA деревьям здесь не нужны.

**Задание:** проверьте дисбаланс и объясните, почему accuracy константы может быть высокой.

Для точного совпадения с соревнованием используем `wCTR (%) > 0`, как в `kaggle/prepare.py`. В одной обучающей строке есть клик, но округлённый wCTR равен нулю; сохраняем официальную метку, не исправляя её по данным test.

In [ ]:
raw = pd.read_csv(DATA_PATH)
dates = pd.to_datetime(raw["Дата"], format="%d.%m.%Y", errors="raise")
y = raw["wCTR (%)"].gt(0).astype(int).rename("target")
cat_columns = ["Группа", "№ Объявления", "Формат", "Размер изображения", "Тип устройства",
               "Пол", "Категория таргетинга", "Упоминание брендов", "Уровень платежеспособности",
               "Возраст", "Заголовок", "Текст"]
X = raw[cat_columns].fillna("не указано").astype(str).copy()
X["Год"] = dates.dt.year
X["Месяц"] = dates.dt.month
X["День недели"] = dates.dt.dayofweek
X["День года"] = dates.dt.dayofyear
assert not set(X.columns) & {"Клики", "CTR (%)", "wCTR (%)", "Расход (руб.)", "Показы"}

print("Размер:", X.shape, "Дней:", dates.nunique())

## 2. Официальный train / test; validation внутри train

Сохраняем **ровно официальные Id**: 8605 строк train и 1318 строк test из `dataset/split.json`. Это те же строки, которые использует `kaggle/prepare.py` для соревнования. Проверяем их против сохранённых `dataset/train.csv` и, если он доступен, `dataset/test.csv`. В чистом клоне без игнорируемого `split.json` список train Id берём из версионируемого `train.csv`, а официальный test — как его дополнение. Новую тестовую выборку не создаём.

Внутри официального train выделяем стратифицированные 20% для validation (`random_state=42`): 6884 строки остаются для обучения, 1721 — для проверки настроек. Из оставшегося train ещё 20% резервируем как `selection`: здесь считаем permutation importance для модели, обученной на `fit`. После отбора обучаем её на всём внутреннем train. Test не входит ни в одну из этих выборок.

Validation используем для подбора параметров и выбора победителя; её оценка поэтому оптимистична. Все кодировки обучаются только на соответствующей обучающей части. Официальный test уже сбалансирован при подготовке соревнования — мы сохраняем его без изменений. Доля кликов и пороговые метрики на нём не отражают естественную частоту кликов в рекламе.

Это официальное разделение по строкам, **не проверка будущих дат**: даты и объявления могут повторяться между train и test. Для переноса на новые даты или объявления нужна отдельная проверка. Метки уже доступны преподавателю в исходной выгрузке и встречались в прежних опытах; здесь воспроизводим оценку курса, не заявляя новый независимый тест.

In [ ]:
# Официальные Id — индексы строк исходного bycicles.csv.
# train.csv хранится в репозитории; split.json — исходный сохранённый манифест.
official_train_file = pd.read_csv(ROOT / "dataset/train.csv", usecols=["Id"])
official_path = ROOT / "dataset/split.json"
if official_path.exists():
    official_split = json.loads(official_path.read_text())
    official_train_idx = pd.Index(official_split["train_indices"])
    test_idx = pd.Index(official_split["test_indices"])
    official_split_source = "dataset/split.json"
else:
    # Тот же официальный test однозначно задаётся дополнением сохранённых train Id.
    official_train_idx = pd.Index(official_train_file.Id)
    test_idx = raw.index.difference(official_train_idx)
    official_split_source = "dataset/train.csv (saved Id; complement for test)"
assert official_train_idx.tolist() == official_train_file.Id.tolist()
assert official_train_idx.is_unique and test_idx.is_unique
assert set(official_train_idx).isdisjoint(test_idx)
assert set(official_train_idx) | set(test_idx) == set(raw.index)
assert len(official_train_idx) == 8605 and len(test_idx) == 1318
if (ROOT / "dataset/test.csv").exists():
    assert test_idx.tolist() == pd.read_csv(ROOT / "dataset/test.csv", usecols=["Id"]).Id.tolist()
# Не выбираем и не генерируем новый test. Обе внутренние выборки только из official train.
train_idx, val_idx = train_test_split(official_train_idx.to_numpy(), test_size=.20,
                                     stratify=y.loc[official_train_idx], random_state=SEED)
train_idx, val_idx = pd.Index(train_idx), pd.Index(val_idx)
fit_idx, selection_idx = train_test_split(train_idx.to_numpy(), test_size=.20,
                                         stratify=y.loc[train_idx], random_state=SEED)
fit_idx, selection_idx = pd.Index(fit_idx), pd.Index(selection_idx)
splits = {"official train (до validation)": official_train_idx,
          "fit (внутри train)": fit_idx, "selection (внутри train)": selection_idx,
          "train": train_idx, "validation": val_idx, "official test": test_idx}
split_table = pd.DataFrame([
    {"выборка": name, "строк": len(idx), "дней": dates.loc[idx].nunique(),
     "начало": dates.loc[idx].min().date(), "конец": dates.loc[idx].max().date()}
    for name, idx in splits.items()
])
display(split_table)
split_table.to_csv(OUT / "splits.csv", index=False)
for idx in [fit_idx, selection_idx, train_idx, val_idx]:
    assert y.loc[idx].nunique() == 2 and set(idx).isdisjoint(test_idx)
assert set(fit_idx).isdisjoint(selection_idx)
assert set(fit_idx) | set(selection_idx) == set(train_idx)
assert set(train_idx).isdisjoint(val_idx)
assert set(train_idx) | set(val_idx) == set(official_train_idx)
display(pd.DataFrame({"строк train": y.loc[train_idx].value_counts(),
                      "доля train": y.loc[train_idx].value_counts(normalize=True)}))
print("Источник официального разделения:", official_split_source)
print("Тестовые ID совпадают с официальными; пересечений с обучением нет.")

## 3. Модель и две метрики

`class_weight="balanced"` увеличивает вклад редкого класса в обучении; строки не удаляем.

- **ROC AUC** оценивает, насколько хорошо модель ранжирует строки с кликом выше строк без клика. Вместе с BalancedAccuracy используется для выбора настроек и общего рейтинга.
- **BalancedAccuracy** — среднее долей верно распознанных строк каждого класса. Считаем при фиксированном пороге 0.5, без подбора по test.

Веса классов считаются только на обучающих данных и не гарантируют откалиброванных вероятностей.

Во всех вариантах есть перебор параметров на validation. Таблицы всех кандидатов сохраняются в `*_search.csv`, сводка выбранных настроек — в `search_summary.csv`. Для RF и ExtraTrees используем общую сетку из 24 комбинаций, для каждого Bagging — 12. В смеси меняем параметры обоих лесов одной комбинацией; веса смеси остаются 50/50.

In [ ]:
def make_model(frame, estimator):
    categorical = [column for column in frame.columns if not pd.api.types.is_numeric_dtype(frame[column])]
    numeric = [column for column in frame.columns if pd.api.types.is_numeric_dtype(frame[column])]
    preprocessing = ColumnTransformer([
        ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                                   dtype=np.float32), categorical),
        ("numeric", "passthrough", numeric),
    ], verbose_feature_names_out=False)
    return Pipeline([("preprocessing", preprocessing), ("model", clone(estimator))])


def metrics(truth, probability):
    assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
    predicted = probability >= .5
    return {"ROC AUC": roc_auc_score(truth, probability),
            "BalancedAccuracy": balanced_accuracy_score(truth, predicted)}


# Вспомогательные функции не обращаются к test.
models, frames, validation_rows = {}, {}, []
search_records = {}
RF_GRID = {"n_estimators": [100, 200], "max_depth": [6, 10, None],
           "min_samples_leaf": [5, 20], "max_features": ["sqrt", .7]}
TREE_GRID = {"max_depth": [3, 5, 10], "min_samples_leaf": [5, 20, 50]}


def register(name, frame, model):
    models[name], frames[name] = model, frame
    score = metrics(y.loc[val_idx], model.predict_proba(frame.loc[val_idx])[:, 1])
    validation_rows.append({"Model": name, "features": frame.shape[1], **score})


def search(name, frame, estimator, grid):
    trials = []
    best_rank, best_model = (-np.inf, -np.inf, -np.inf), None
    for params in ParameterGrid(grid):
        start = time.perf_counter()
        candidate = make_model(frame, clone(estimator).set_params(**params))
        candidate.fit(frame.loc[train_idx], y.loc[train_idx])
        score = metrics(y.loc[val_idx], candidate.predict_proba(frame.loc[val_idx])[:, 1])
        auc = score["ROC AUC"]
        rank = ((auc + score["BalancedAccuracy"]) / 2, auc, score["BalancedAccuracy"])
        trials.append({**params, "validation_score": rank[0], "validation_auc": auc,
                       "validation_BalancedAccuracy": score["BalancedAccuracy"], "seconds": time.perf_counter() - start})
        if rank > best_rank:
            best_rank, best_model = rank, candidate
    table = pd.DataFrame(trials).sort_values(["validation_score", "validation_auc", "validation_BalancedAccuracy"], ascending=False, kind="stable")
    filename = name.replace("/", "_")
    table.to_csv(OUT / f"{filename}_search.csv", index=False)
    display(table.head(5))
    search_records[name] = {"trials": len(trials), "grid": grid,
                            "best_params": table.iloc[0].to_dict()}
    register(name, frame, best_model)
    return best_model

## 4. DecisionTree

**Задание:** сравните глубину 3, 5, 10 и размер листа 5, 20, 50. Выберите по среднему validation ROC AUC и BalancedAccuracy.
Почему маленький лист может запоминать редкие сочетания категорий?

In [ ]:
# TODO: Обучите дерево через search("DecisionTree", X, estimator, grid); сохраните pipeline в tree.
raise NotImplementedError("Выполните задание выше; полный вариант — в solution")

## 5. RandomForest

Лес усредняет вероятности многих деревьев. Каждое видит bootstrap-выборку строк,
а в каждом разбиении рассматривается случайная часть признаков. Это уменьшает зависимость деревьев
и разброс прогноза, но не устраняет утечки и сдвиг распределения.

**Задание:** переберите 100 и 200 деревьев, сравните `max_depth`, `min_samples_leaf`, `max_features`.
Для каждого леса проверяем общую сетку: 100/200 деревьев, глубина 6/10/без ограничения, лист 5/20, max_features sqrt/0.7 — 24 комбинации.

In [ ]:
# TODO: Обучите лес через search("RandomForest", X, estimator, grid); сохраните pipeline в forest.
raise NotImplementedError("Выполните задание выше; полный вариант — в solution")

## 6. Permutation importance → отбор → переобучение

Перемешиваем один **исходный** столбец перед pipeline: все его one-hot категории меняются вместе.
Падение ROC AUC измеряем на `selection`, а модель для этого шага учим только на `fit`.
Порог фиксируем заранее: среднее падение должно превышать одно стандартное отклонение по перемешиваниям.
Это эвристика устойчивости, а не статистический доверительный интервал. Если ни один столбец не прошёл,
сохраняем один с максимальным средним вкладом и явно сообщаем об этом.

Связанные признаки могут заменять друг друга, поэтому важность бывает занижена.
Permutation importance не «исправляет корреляции», а отбор не обязан повышать качество.
После отбора заново подбираем параметры дерева и леса на validation; набор полей фиксируется по selection и не меняется при переборе.

**Задание:** реализуйте отбор отдельно для дерева и леса, переобучите на полном train,
затем сравните с исходными моделями на validation.

In [ ]:
# TODO: Создайте importance_tables и функцию select_by_permutation; зарегистрируйте оба варианта с суффиксом " + permutation". См. протокол в тексте.
raise NotImplementedError("Выполните задание выше; полный вариант — в solution")

### Итог отбора признаков

Сравним модели на всех исходных полях и после отбора. В таблице показаны фактически используемые признаки и обе метрики на validation. Отбор выполнен на selection внутри train; test остаётся для раздела 9.

In [ ]:
# Итог отбора: те же train/validation, только два показателя качества.
selection_order = ["DecisionTree", "DecisionTree + permutation",
                   "RandomForest", "RandomForest + permutation"]
selection_scores = pd.DataFrame(validation_rows).set_index("Model")
selection_summary = pd.DataFrame([
    {"Модель": name,
     "Признаков": frames[name].shape[1],
     "Используемые признаки": ", ".join(frames[name].columns),
     "Val ROC AUC": selection_scores.loc[name, "ROC AUC"],
     "Val BalancedAccuracy": selection_scores.loc[name, "BalancedAccuracy"]}
    for name in selection_order
]).set_index("Модель")
with pd.option_context("display.max_colwidth", None):
    display(selection_summary.round(4))
selection_summary.to_csv(OUT / "feature_selection_summary.csv")
for name in ["DecisionTree", "RandomForest"]:
    selected_columns = importance_tables[name].loc[importance_tables[name].selected, "feature"].tolist()
    assert frames[name + " + permutation"].columns.tolist() == selected_columns
    assert selection_summary.loc[name + " + permutation", "Признаков"] == len(selected_columns)
assert selection_summary.index.tolist() == selection_order
print("Тест не используется в этой таблице; итоговые test-метрики — в разделе 9.")

### 6.2. Подвыборки признаков: Random Subspace и Random Patches

В обычном Random Forest случайные признаки выбираются заново **в каждом узле**. Здесь каждому дереву один раз назначаем набор столбцов; все его узлы используют только этот набор (`BaggingClassifier.max_features`). У самого `DecisionTreeClassifier` задаём `max_features=None`, чтобы дополнительного отбора в узлах не было.

Проверяем:

- **Random Subspace:** все строки train, случайные 50% или 75% столбцов для каждого дерева.
- **Random Patches:** bootstrap строк и случайные 50% или 75% столбцов для каждого дерева.
- **Контроль:** bootstrap строк, все столбцы; это bagging без отбора признаков.

Подвыборки выбираются **после one-hot кодирования**: одна категория может попасть в дерево, а другая категория того же поля — нет. Это доли кодированных столбцов, а не исходных 16 полей. Кодировщик учится только на train. Для каждого ансамбля с `class_weight="balanced"` перебираем число деревьев 100/200, глубину 6/10/без ограничения и размер листа 5/20 — 12 комбинаций. Доля признаков задаётся вариантом, а внутри дерева `max_features=None`. Среди четырёх вариантов с подпространствами выбираем лучший только по validation. Здесь показываем validation-таблицу; test-результаты для всех ансамблей появятся в разделе 9.

Используем те же официальные Id и внутренние train/validation. Обучение и выбор подвыборки признаков выполняем до раздела Kaggle; официальные test-строки не участвуют в выборе настроек.

**Задание:** обучите пять ансамблей, сравните их с базовым RandomForest. Проверьте через `estimators_features_`, что наборы столбцов у деревьев различаются, а внутри дерева доступны все назначенные ему признаки.

[Документация sklearn: BaggingClassifier, Random Subspaces и Random Patches](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html).


In [ ]:
# TODO: обучите Random Subspace 50%/75%, Random Patches 50%/75% и bagging-контроль.
# Для каждого ансамбля переберите число деревьев, глубину и размер листа; сохраните search_records.
# Зарегистрируйте лучший pipeline через register, чтобы он участвовал в общем выборе.
# Сохраните subspace_models, subspace_encoder, encoded_names, subspace_tree, settings,
# subspace_validation и subspace_best. Здесь считайте только validation-метрики.
raise NotImplementedError("Подвыборки признаков до Kaggle")

## 7. Пробуем интересные решения из Kaggle

| Ссылка на решение | Идея в нём | Как адаптируем |
|---|---|---|
| [Heart Disease, 1-е место — февраль 2026](https://www.kaggle.com/competitions/playground-series-s6e2/writeups/1st-place-solution-diversity-selection-and-t) | Добавить частоты категорий к признакам деревьев | **RF + frequency:** частоты группы, объявления, таргетинга, заголовка и текста считаем только на train; неизвестным значениям даём 0 |
| [Podcast Listening Time, 3-е место — май 2025](https://www.kaggle.com/competitions/playground-series-s5e4/writeups/johannes-heller-3rd-place-target-encoding-and-3-le) | Средняя целевая по категории как дополнительный признак; в ансамбле есть RF и ExtraTrees | **RF + target encoding:** добавляем сглаженную долю кликов по четырём категориальным полям; обучающие значения считаем по внутренним фолдам |
| [Optimal Fertilizers, 1-е место — июль 2025](https://www.kaggle.com/competitions/playground-series-s5e6/writeups/chris-deotte-1st-place-fast-gpu-experimentation-wi) | Target encoding сочетаний двух и более признаков для деревьев | **RF + pair target encoding:** кодируем три пары: группа × таргетинг, устройство × возраст, группа × месяц; используем пять внутренних фолдов |
| [Customer Churn, 18-е место — апрель 2026](https://www.kaggle.com/competitions/playground-series-s6e3/writeups/18th-place-many-oofs-neural-networks-over-gbdts) | Сравнить RF и ExtraTrees с разной случайностью разбиений | **ExtraTrees:** вместо оптимального порога пробуем случайные пороги; число деревьев, глубину, лист и max_features подбираем по общей сетке |
| [Road Accident Risk, 14-е место — октябрь 2025](https://www.kaggle.com/competitions/playground-series-s5e10/writeups/s5e10-14th-place-solution) | Усреднять прогнозы разных ансамблей, включая RF и ExtraTrees | **RF + ExtraTrees blend:** простое среднее вероятностей двух лесов с весами 50/50, с подбором параметров обоих лесов; веса 50/50 фиксированы |


In [ ]:
# TODO: реализуйте пять приёмов из таблицы и сохраните прежние эксперименты. Для каждого выполните search с RF_GRID; у смеси подбирайте параметры обоих лесов.
# Создайте kaggle_experiments, experiments, validation; обучите и зарегистрируйте модели.
# TargetEncoder: fit_transform на train с внутренними фолдами, transform на validation/test.
raise NotImplementedError("Пять приёмов Kaggle")

### 7.2. Ещё один поиск: другие представления признаков

| Ссылка на решение | Идея в нём | Как адаптируем |
|---|---|---|
| [Customer Churn, 17-е место — март 2026](https://www.kaggle.com/competitions/playground-series-s6e3/writeups/17th-place-solution) | Периодические sin/cos представления | Добавляем циклические день недели, месяц и день года |
| [То же решение](https://www.kaggle.com/competitions/playground-series-s6e3/writeups/17th-place-solution) | CATNUM: числовые распределения внутри категорий без целевой метки | Среднее и разброс календарных признаков, число разных объявлений внутри группы и таргетинга; статистики только train |
| [То же решение](https://www.kaggle.com/competitions/playground-series-s6e3/writeups/17th-place-solution) | Частоты сочетаний категорий без target encoding | Частоты четырёх пар: объявление × устройство, группа × возраст, группа × таргетинг, устройство × возраст |
| [Heart Disease, 1-е место — февраль 2026](https://www.kaggle.com/competitions/playground-series-s6e2/writeups/1st-place-solution-diversity-selection-and-t) | Binning и представление чисел как категорий | Дублируем месяц, день недели и недельный интервал дня года как категории |
| [Irrigation Need, 1-е место — апрель 2026](https://www.kaggle.com/competitions/playground-series-s6e4/writeups/1st-place-one-vs-rest-approach) | Выбирать правило классификации под BalancedAccuracy на валидационных прогнозах | В разделе 13 выбираем порог по validation, не меняя вероятности и ROC AUC |

Четыре представления сравниваем с контрольным RF: у всех общая RF_GRID по числу деревьев, глубине, размеру листа и max_features. Это позволяет отделить влияние новых признаков от дополнительного подбора параметров. Все леса используют `class_weight="balanced"`.


In [ ]:
# TODO: реализуйте четыре представления без использования target.
# Сравните с RF retuned control на общей RF_GRID, обновите experiments и validation.
raise NotImplementedError("Второй поиск признаков")

## 8. Общий лидерборд на validation

Рейтинг = **(ROC AUC + BalancedAccuracy) / 2**: равные веса двух метрик. Для каждого метода берём комбинацию гиперпараметров с максимальным рейтингом. При равенстве выше ROC AUC, затем BalancedAccuracy. Порог классификации — 0.5.

In [ ]:
# Для каждого метода выбираем лучшую комбинацию по среднему двух validation-метрик.
leaderboard_rows = []
for name, record in search_records.items():
    trials = pd.read_csv(OUT / (name.replace("/", "_") + "_search.csv"))
    trials["Validation score"] = (trials["validation_auc"] + trials["validation_BalancedAccuracy"]) / 2
    best = trials.sort_values(["Validation score", "validation_auc", "validation_BalancedAccuracy"],
                              ascending=False, kind="stable").iloc[0]
    parameter_names = list(next(iter(ParameterGrid(record["grid"]))))
    parameters = []
    for key in parameter_names:
        value = best[key]
        if pd.isna(value):
            label = "без ограничения" if key.endswith("max_depth") else "None"
        elif isinstance(value, (int, float, np.integer, np.floating)):
            label = str(int(value)) if float(value).is_integer() else str(value)
        else:
            label = str(value)
        parameters.append(f"{key}={label}")
    leaderboard_rows.append({"Метод": name, "Среднее метрик": best["Validation score"],
                             "Val ROC AUC": best["validation_auc"],
                             "Val BalancedAccuracy": best["validation_BalancedAccuracy"],
                             "Лучшие параметры": "; ".join(parameters)})
validation_leaderboard = pd.DataFrame(leaderboard_rows).sort_values(
    ["Среднее метрик", "Val ROC AUC", "Val BalancedAccuracy"], ascending=False, kind="stable")
validation_leaderboard.index = pd.RangeIndex(1, len(validation_leaderboard) + 1, name="Место")
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(validation_leaderboard.round(4))
validation_leaderboard.to_csv(OUT / "validation_leaderboard.csv")
assert validation_leaderboard["Метод"].is_unique
assert set(validation_leaderboard["Метод"]) == set(search_records)
assert validation_leaderboard["Среднее метрик"].is_monotonic_decreasing
np.testing.assert_allclose(validation_leaderboard["Среднее метрик"],
                           validation_leaderboard[["Val ROC AUC", "Val BalancedAccuracy"]].mean(axis=1))

### Фиксируем настройки до test

Для каждого метода оставляем лучшую комбинацию по среднему двух validation-метрик — ту же, что указана в лидерборде выше. Эти модели оцениваем на test в разделе 9. Порог основной таблицы фиксирован: 0.5. Гиперпараметры на test не подбираем.

In [ ]:
CORE = ["DecisionTree", "DecisionTree + permutation", "RandomForest", "RandomForest + permutation"]
validation_best_name = validation_leaderboard.iloc[0]["Метод"]
best_name = validation_best_name
print("Лидер validation:", validation_best_name)
protocol = {"seed": SEED, "data_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
            "target": "wCTR (%) > 0 (official competition target)", "hyperparameters_selected_on": "validation mean(ROC AUC, BalancedAccuracy)",
            "validation_winner": validation_best_name, "winner_selected_on": "validation mean(ROC AUC, BalancedAccuracy)", "winner": best_name,
            "threshold": .5, "metrics": ["ROC AUC", "BalancedAccuracy"],
            "target_encoding": {"fit_ids": train_idx.tolist(), "cv": 5, "smooth": 20, "cross_fitting": True}, "permutation_selected_on": "selection (inside train)",
            "official_split_source": official_split_source,
            "official_train_indices": official_train_idx.tolist(),
            "official_test_indices": test_idx.tolist(),
            "official_ids_sha256": hashlib.sha256(json.dumps({"train": official_train_idx.tolist(), "test": test_idx.tolist()}).encode()).hexdigest(),
            "python": platform.python_version(), "sklearn": sklearn.__version__,
            "train_indices": train_idx.tolist(), "validation_indices": val_idx.tolist(),
            "test_indices": test_idx.tolist(), "fit_indices": fit_idx.tolist(),
            "selection_indices": selection_idx.tolist(),
            "models": {name: {"columns": frames[name].columns.tolist(),
                               "params": model.named_steps["model"].get_params()}
                       for name, model in models.items()}}
(OUT / "protocol.json").write_text(json.dumps(protocol, ensure_ascii=False, indent=2, default=str) + "\n")
assert set(search_records) == set(models)
search_summary = pd.DataFrame([
    {"Model": name, "trials": record["trials"], "best_params": json.dumps(record["best_params"], ensure_ascii=False, default=str)}
    for name, record in search_records.items()
]).set_index("Model")
search_summary.to_csv(OUT / "search_summary.csv")
(OUT / "search_protocol.json").write_text(json.dumps(search_records, ensure_ascii=False, indent=2, default=str) + "\n")
display(search_summary)


## 9. Общий лидерборд на test

Те же методы и настройки, выбранные в разделе 8. Рейтинг = **(Test ROC AUC + Test BalancedAccuracy) / 2**. Сортируем по нему, при равенстве — по ROC AUC, затем BalancedAccuracy. В конце тетради SHAP объясняет лидера этой таблицы; это разбор уже просмотренного test, а не новая независимая проверка.

In [ ]:
assert y.loc[test_idx].nunique() == 2
rows = []
predictions = pd.DataFrame({"row_id": test_idx, "date": dates.loc[test_idx].astype(str).to_numpy(),
                            "target": y.loc[test_idx].to_numpy()})
for name, model in models.items():
    probability = model.predict_proba(frames[name].loc[test_idx])[:, 1]
    predictions[name] = probability
    rows.append({"Model": name, "Val ROC AUC": validation.loc[name, "ROC AUC"],
                 "Val BalancedAccuracy": validation.loc[name, "BalancedAccuracy"],
                 **{"Test " + key: value for key, value in metrics(y.loc[test_idx], probability).items()}})
results = pd.DataFrame(rows).set_index("Model")
comparison = results.loc[CORE]
test_leaderboard = validation_leaderboard[["Метод", "Лучшие параметры"]].copy()
for metric in ["ROC AUC", "BalancedAccuracy"]:
    test_leaderboard["Test " + metric] = test_leaderboard["Метод"].map(results["Test " + metric])
test_leaderboard["Среднее метрик"] = test_leaderboard[["Test ROC AUC", "Test BalancedAccuracy"]].mean(axis=1)
test_leaderboard = test_leaderboard[["Метод", "Среднее метрик", "Test ROC AUC", "Test BalancedAccuracy", "Лучшие параметры"]].sort_values(
    ["Среднее метрик", "Test ROC AUC", "Test BalancedAccuracy"], ascending=False, kind="stable")
test_leaderboard.index = pd.RangeIndex(1, len(test_leaderboard) + 1, name="Место")
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(test_leaderboard.round(4))
test_leaderboard.to_csv(OUT / "test_leaderboard.csv")
best_name = test_leaderboard.iloc[0]["Метод"]
protocol.update(winner=best_name, winner_selected_on="test mean(ROC AUC, BalancedAccuracy) — exploratory SHAP",
                shap_model=best_name)
(OUT / "protocol.json").write_text(json.dumps(protocol, ensure_ascii=False, indent=2, default=str) + "\n")
print("Модель для заключительного SHAP:", best_name)
comparison.to_csv(OUT / "comparison.csv")
results.to_csv(OUT / "all_results.csv")
predictions.to_csv(OUT / "test_predictions.csv", index=False)

### Итог подвыборок признаков на test

Теперь проверяем ансамбли, обученные в разделе 6.2. Сохраняем общую таблицу со всеми вариантами.

In [ ]:
# TODO: оцените обученные subspace_models на официальном test.
# Сохраните subspace_results.csv и extended_comparison.csv.
raise NotImplementedError("Итоговая проверка подвыборок признаков")

In [ ]:
for base in ["DecisionTree", "RandomForest"]:
    delta = results.loc[base + " + permutation", "Test ROC AUC"] - results.loc[base, "Test ROC AUC"]
    print(f"{base}: изменение test ROC AUC после отбора {delta:+.4f}")
for name, _, _ in experiments:
    delta = results.loc[name, "Test ROC AUC"] - results.loc["RandomForest", "Test ROC AUC"]
    print(f"{name}: изменение test ROC AUC относительно базового леса {delta:+.4f}")
print("Разницы на одном официальном test — наблюдение, а не доказательство устойчивого превосходства.")

probability = predictions[best_name].to_numpy()
RocCurveDisplay.from_predictions(y.loc[test_idx], probability)
plt.title(best_name)
plt.tight_layout(); plt.savefig(OUT / "winner_metrics.png", dpi=140); plt.show()

### Ваш разбор результата

Сформулируйте вывод по четырём строкам сравнения и дополнительным опытам. Назовите три главных исходных признака по SHAP, объясните направление вкладов на beeswarm и разберите два локальных примера. Укажите, где модель ошиблась, и почему SHAP не доказывает причинность. Победителя после просмотра test не меняйте.

## 10. Порог для BalancedAccuracy

ROC AUC зависит от вероятностей, а BalancedAccuracy — ещё и от порога. Для лидера validation, сравним фиксированный порог 0.5 и порог, максимизирующий validation BalancedAccuracy в заранее заданной сетке 0.10–0.90 с шагом 0.01. При равенстве выбираем ближайший к 0.5.

Дополнительную поправку на доли классов не применяем: лес уже обучается с весами классов. ROC AUC и SHAP вероятностей от смены порога не меняются. Новую BalancedAccuracy показываем отдельной таблицей, чтобы не смешивать её с исходным сравнением при пороге 0.5.

In [ ]:
# TODO: выберите порог в сетке 0.10–0.90 по validation BalancedAccuracy.
# Сравните с 0.5; test для выбора порога не используйте.
raise NotImplementedError("Порог для BalancedAccuracy")

## 11. Проверяем результат и обсуждаем

1. В каких строках отбор помог или навредил? Сколько признаков осталось?
2. Совпали ли победители validation и test? Почему выбор лидера test для объяснения нельзя считать независимым подтверждением его качества?
3. Какие новые признаки и настройки леса оказались полезны на этом официальном разделении?
4. Какие исходные поля сильнее влияют на прогноз лучшей модели? Совпадают ли глобальные и локальные объяснения?
5. Почему следующий эксперимент лучше проверять на новых данных или на нескольких временных окнах?

Проверки ниже выполняются вместе с тетрадью. CSV, протокол и графики сохраняются в `seminars/trees_results/`.

## Источники

- [Решение Kaggle 1](https://www.kaggle.com/competitions/playground-series-s6e2/writeups/1st-place-solution-diversity-selection-and-t).
- [Решение Kaggle 2](https://www.kaggle.com/competitions/playground-series-s5e4/writeups/johannes-heller-3rd-place-target-encoding-and-3-le).
- [Решение Kaggle 3](https://www.kaggle.com/competitions/playground-series-s5e6/writeups/chris-deotte-1st-place-fast-gpu-experimentation-wi).
- [Решение Kaggle 4](https://www.kaggle.com/competitions/playground-series-s6e3/writeups/18th-place-many-oofs-neural-networks-over-gbdts).
- [Решение Kaggle 5](https://www.kaggle.com/competitions/playground-series-s5e10/writeups/s5e10-14th-place-solution).

- [sklearn: TargetEncoder и cross-fitting](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html).
- [sklearn: ExtraTreesClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html).
- [SHAP: TreeExplainer](https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html).

## 12. SHAP лидера тестового рейтинга

Объясняем модель с наибольшим средним ROC AUC и BalancedAccuracy в разделе 9. Её гиперпараметры ранее выбраны только на validation. Глобальные графики показывают силу и направление вкладов; два локальных примера — строка с кликом и без клика.

Для Bagging возвращаем вклады каждого дерева в общую систему признаков и усредняем. Для смеси RF и ExtraTrees усредняем объяснения с весами 50/50. Проверяем, что базовое значение плюс сумма SHAP равны вероятности класса 1. SHAP объясняет поведение модели, а не причинное влияние.

In [ ]:
# TODO: Объясните выбранную модель через TreeExplainer; создайте positive (класс 1), preprocessing, X_winner, explain_idx и local_idx. Проверьте аддитивность.
raise NotImplementedError("Выполните задание выше; полный вариант — в solution")

In [ ]:
# TODO: Сгруппируйте one-hot SHAP по исходным признакам; покажите глобальную важность и два локальных waterfall. Сохраните графики в OUT.
raise NotImplementedError("Выполните задание выше; полный вариант — в solution")

In [ ]:
assert comparison.index.tolist() == CORE and comparison.shape[0] == 4
assert validation_best_name == validation_leaderboard.iloc[0]["Метод"]
assert best_name == test_leaderboard.iloc[0]["Метод"]
assert set(fit_idx).isdisjoint(selection_idx)
assert set(fit_idx) | set(selection_idx) == set(train_idx)
assert set(train_idx).isdisjoint(val_idx) and set(train_idx).isdisjoint(test_idx)
assert set(val_idx).isdisjoint(test_idx)
for name, model in models.items():
    fitted = model.named_steps["model"]
    weighted_models = fitted.estimators_ if isinstance(fitted, (VotingClassifier, BaggingClassifier)) else [fitted]
    assert all(m.class_weight in ("balanced", "balanced_subsample") for m in weighted_models)
    assert model.named_steps["preprocessing"].n_features_in_ == frames[name].shape[1]
    np.testing.assert_allclose(results.loc[name, "Test ROC AUC"],
                               roc_auc_score(predictions.target, predictions[name]))
for name, table in importance_tables.items():
    assert set(frames[name + " + permutation"].columns) == set(table.loc[table.selected, "feature"])
assert np.isfinite(positive.values).all()
assert (OUT / "shap_beeswarm.png").is_file()
print("Проверки разделения, четырёх моделей, весов, метрик и SHAP пройдены.")
assert set(train_idx) | set(val_idx) == set(official_train_idx)
assert protocol["official_test_indices"] == test_idx.tolist()
assert set(official_train_idx).isdisjoint(test_idx)
assert set(metrics(y.loc[test_idx], predictions[best_name])) == {"ROC AUC", "BalancedAccuracy"}
assert len(kaggle_experiments) == 5
assert set(search_records) == set(models)
assert all(record["trials"] > 1 for record in search_records.values())